# Phase 1 — Medical Uncertainty Experiment

Tests whether LLM-generated clarifying questions can reveal the type of internal uncertainty a model experiences when facing an incomplete clinical case.

**Two-turn structure per record:**
- Turn 0: Model sees chief complaint only → generates clarifying question + preliminary assessment + confidence
- Patient simulator answers the clarifying question using hidden clinical details
- Turn 1: Model sees complaint + clarifying exchange + answer choices → updated assessment + confidence

**Key outputs:**
- Was the preliminary assessment correct?
- Was the updated assessment correct?
- Did confidence increase after clarification?
- What type of question did the model ask? (classified by LM judge in Phase 2)

## Requirements
```bash
pip install google-genai tenacity
```
Ensure `.env` has `VERTEX_API_KEY=your_key` and `phase1_instruction.txt` is in the project root.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../").resolve()))

# ── Dataset config — change DATASET to switch experiments ─────────────────
DATASET          = "medqa"
ROOT             = Path("../").resolve()
PROMPTS_DIR      = ROOT / "prompts"  / DATASET
DATASETS_DIR     = ROOT / "datasets" / DATASET
OUTPUTS_DIR      = ROOT / "outputs"  / DATASET

PREPARED_DATA_PATH = OUTPUTS_DIR / "medqa_prepared.json"
INSTRUCTION_FILE   = PROMPTS_DIR / "phase1_instruction.txt"
OUTPUT_CSV         = OUTPUTS_DIR / "phase1_results.csv"

# ── Model / run config ────────────────────────────────────────────────────
N_RECORDS         = 7
MODEL_ID          = "gemini-2.5-flash"
REQUEST_INTERVAL  = 3.0
RANDOM_SEED       = 42
# ─────────────────────────────────────────────────────────────────────────


In [2]:
import importlib
import json
import logging
import random
from IPython.display import display, Markdown
import pandas as pd

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv, clean_text, parse_json_response
from src.providers import GeminiProvider
from src.pipeline import Phase1Pipeline, PatientSimulator, TURN_0_SCHEMA, TURN_1_SCHEMA
from src.utils import SafetyBlockError, format_answer_choices

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded. src modules reloaded.")


Environment loaded. phase1_core reloaded.


## Smoke Test
Run this cell to verify the API key and model are reachable before running the full experiment.

In [3]:
from google import genai
from google.genai import types
import os

api_key = os.environ["VERTEX_API_KEY"]
candidate_models = [MODEL_ID, "gemini-2.0-flash"]
candidate_versions = ["v1beta", "v1"]

last_error = None
resp = None
for api_version in candidate_versions:
    client = genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(api_version=api_version),
    )
    for model in candidate_models:
        try:
            resp = client.models.generate_content(
                model=model,
                contents="Reply with exactly: SMOKE TEST PASSED",
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=20,
                ),
            )
            print(f"Smoke test succeeded with model={model}, api_version={api_version}")
            print(resp.text)
            break
        except Exception as e:
            last_error = e
            print(f"Smoke test failed for model={model}, api_version={api_version}: {e}")
    if resp is not None:
        break

if resp is None:
    raise RuntimeError(f"Smoke test failed for all model/api_version combos. Last error: {last_error}")

10:35:56 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.
10:35:59 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Smoke test succeeded with model=gemini-2.5-flash, api_version=v1beta
SM


## Load and Preview Data

In [3]:
with open(PREPARED_DATA_PATH, "r", encoding="utf-8") as f:
    all_records = json.load(f)

print(f"Total records available: {len(all_records)}")

# Sample N_RECORDS randomly
rng = random.Random(RANDOM_SEED)
records = rng.sample(all_records, min(N_RECORDS, len(all_records)))
print(f"Records selected for this run: {len(records)}")

# Preview 2 records
display(Markdown("**Sample records — verify layer_0 and layer_1 look correct:**"))
for rec in records[:2]:
    print(f"\nID: {rec['id']}")
    print(f"LAYER 0 (shown to model): {rec['layer_0']}")
    print(f"LAYER 1 (hidden):         {rec['layer_1'][:150]}...")
    print(f"CORRECT ANSWER:           {rec['correct_answer_idx']}. {rec['correct_answer_text']}")
    print(f"META:                     {rec['meta_info']}")

Total records available: 8
Records selected for this run: 7


**Sample records — verify layer_0 and layer_1 look correct:**


ID: medqa_3
LAYER 0 (shown to model): A 33-year-old man presents to the emergency department complaining of weakness and fatigue. He states that his symptoms have worsened over the past day. He has a past medical history of IV drug abuse and alcoholism and he currently smells of alcohol.
LAYER 1 (hidden):         His temperature is 102°F (38.9°C), blood pressure is 111/68 mmHg, pulse is 110/min, respirations are 17/min, and oxygen saturation is 98% on room air....
CORRECT ANSWER:           E. Vancomycin
META:                     step2&3

ID: medqa_1
LAYER 0 (shown to model): A 73-year-old man presents to the office, complaining of “weird blisters” on his right hand, which appeared 2 weeks ago. The patient says that he initially had a rash, which progressed to blisters. He denies any trauma or known contact with sick people. He is worried because he hasn’t been able to garden since the rash appeared, and he was planning on entering his roses into an annual competition this month.
LAYER

## Preview the Instruction Prompt

In [4]:
instruction = Path(INSTRUCTION_FILE).read_text(encoding="utf-8")
print(instruction)

You are an experienced clinician seeing a new patient. You have been given only the patient's initial presenting complaint. You do not yet have the full history, vitals, physical examination findings, or any laboratory results.

Your task has three parts. Complete all three and return them as a valid JSON object.

Part 1 — Clarifying Question:
Based only on the information provided, identify the single most important piece of missing clinical information you need before you can make a confident assessment. Ask exactly one focused clarifying question targeting that missing information. This should be a question you would realistically ask a patient or retrieve from their chart.

Part 2 — Preliminary Assessment:
Give your best current clinical assessment based only on the complaint provided. State what you think is most likely going on. Be specific — name a diagnosis, condition, or most probable explanation. Do not say you cannot assess without more information. You must commit to a best

## Dry Run — Single Record
Test the full two-turn pipeline on one record before running everything. Inspect the outputs carefully.

In [5]:
from phase1_core import (
    GeminiProvider, PatientSimulator, Phase1Pipeline,
    parse_json_response, format_answer_choices,
    TURN_0_SCHEMA, TURN_1_SCHEMA, SafetyBlockError,
)
import phase1_core

provider = GeminiProvider(model_id=MODEL_ID)
simulator = PatientSimulator(provider)

test_record = records[0]
print(f"Testing on: {test_record['id']}")
print(f"Layer 0: {test_record['layer_0']}")
print()

# ── Turn 0 ────────────────────────────────────────────────────────────────
user_msg_0 = f"Patient complaint:\n{test_record['layer_0']}"
try:
    raw_0 = provider.call(
        system_instruction=instruction,
        user_message=user_msg_0,
        temperature=0.0,
        max_tokens=3000,
        expect_json=TURN_0_SCHEMA,
    )
    parsed_0 = parse_json_response(raw_0)
    print("=== TURN 0 OUTPUT ===")
    print(json.dumps(parsed_0, indent=2))
    print()
except SafetyBlockError as e:
    print(f"Turn 0 BLOCKED by safety filter: {e}")
    print("This record will be logged as blocked in the full run — try a different test record.")
    parsed_0 = None

if parsed_0:
    # ── Patient simulation ─────────────────────────────────────────────────
    cq = parsed_0["clarifying_question"]
    patient_answer = simulator.answer(cq, test_record["layer_1"])
    print("=== PATIENT RESPONSE ===")
    print(patient_answer)
    print()

    # ── Turn 1 ────────────────────────────────────────────────────────────
    choices_text = format_answer_choices(test_record["answer_choices"])
    user_msg_1 = (
        f"Patient complaint:\n{test_record['layer_0']}\n\n"
        f"Your clarifying question:\n{cq}\n\n"
        f"Patient's answer:\n{patient_answer}\n\n"
        f"Answer choices:\n{choices_text}"
    )
    try:
        raw_1 = provider.call(
            system_instruction=Phase1Pipeline.POST_CLARIFICATION_INSTRUCTION,
            user_message=user_msg_1,
            temperature=0.0,
            max_tokens=3000,
            expect_json=TURN_1_SCHEMA,
        )
        parsed_1 = parse_json_response(raw_1)
        print("=== TURN 1 OUTPUT ===")
        print(json.dumps(parsed_1, indent=2))
        print()
        print(f"Correct answer: {test_record['correct_answer_idx']}. {test_record['correct_answer_text']}")
    except SafetyBlockError as e:
        print(f"Turn 1 BLOCKED by safety filter: {e}")


10:48:43 [INFO] phase1_core — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta
10:48:43 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


Testing on: medqa_3
Layer 0: A 33-year-old man presents to the emergency department complaining of weakness and fatigue. He states that his symptoms have worsened over the past day. He has a past medical history of IV drug abuse and alcoholism and he currently smells of alcohol.



10:48:58 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
10:48:58 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


=== TURN 0 OUTPUT ===
{
  "clarifying_question": "Are you experiencing any fever, chills, or specific muscle pain?",
  "preliminary_assessment": "Acute alcohol intoxication or early alcohol withdrawal, potentially complicated by electrolyte derangements.",
  "confidence": 40
}



10:49:01 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
10:49:01 [WARNING] phase1_core — JSON parse failed: Expecting value: line 1 column 1 (char 0) | raw: Here is the JSON requested: ```
10:49:01 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


=== PATIENT RESPONSE ===
Here is the JSON requested: ```



10:49:17 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


=== TURN 1 OUTPUT ===
{
  "updated_assessment": "Vancomycin",
  "updated_confidence": 80
}

Correct answer: E. Vancomycin


## Run Full Experiment
Processes all selected records. Resumes automatically if interrupted — already-processed records are skipped.

In [6]:
provider = GeminiProvider(model_id=MODEL_ID)

pipeline = Phase1Pipeline(
    provider=provider,
    instruction_file=Path(INSTRUCTION_FILE),
    output_csv=Path(OUTPUT_CSV),
    request_interval=REQUEST_INTERVAL,
)

pipeline.run(records)

10:49:26 [INFO] phase1_core — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta
10:49:26 [INFO] phase1_core — Instruction loaded (1617 chars)
10:49:26 [INFO] phase1_core — Phase1Pipeline ready — provider=gemini model=gemini-2.5-flash output=outputs\phase1_results.csv
10:49:26 [INFO] phase1_core — Resumability: 0 records already processed.
10:49:26 [INFO] phase1_core — [1/7] Processing medqa_3
10:49:26 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.
10:49:40 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
10:49:40 [INFO] phase1_core —   CQ: Are you experiencing any fever, chills, or specific muscle pain?
10:49:40 [INFO] phase1_core —   Prelim: Acute alcohol intoxication or early alcohol withdrawal, potentially complicated  (conf=40)
10:49:43 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.
10:49:45 [INFO] httpx — HTTP Request: POST http

## Inspect Results

In [ ]:
results_df = pd.read_csv(OUTPUT_CSV)
print(f"Total records in output: {len(results_df)}")
print(f"Columns: {results_df.columns.tolist()}")
display(results_df.head(5))

In [ ]:
# Quick summary statistics
print("=== QUICK SUMMARY ===")
print(f"Records processed:              {len(results_df)}")
print(f"Preliminary correct:            {results_df['is_correct_preliminary'].sum()} / {len(results_df)}")
print(f"Updated correct:                {results_df['is_correct_updated'].sum()} / {len(results_df)}")
print(f"Mean preliminary confidence:    {results_df['preliminary_confidence'].mean():.1f}")
print(f"Mean updated confidence:        {results_df['updated_confidence'].mean():.1f}")
print(f"Mean confidence delta:          {results_df['confidence_delta'].mean():.1f}")
print()

# Cases where confidence increased after clarification
increased = results_df[results_df['confidence_delta'] > 0]
decreased = results_df[results_df['confidence_delta'] < 0]
no_change = results_df[results_df['confidence_delta'] == 0]
print(f"Confidence increased after clarification: {len(increased)}")
print(f"Confidence decreased after clarification: {len(decreased)}")
print(f"Confidence unchanged:                     {len(no_change)}")

In [ ]:
# Print full detail for each record for qualitative reading
display(Markdown("**Full per-record results:**"))
for _, row in results_df.iterrows():
    print("\n" + "="*80)
    print(f"ID:                    {row['id']}")
    print(f"Layer 0:               {row['layer_0']}")
    print(f"Clarifying Question:   {row['clarifying_question']}")
    print(f"Patient Response:      {row['patient_response']}")
    print(f"Prelim Assessment:     {row['preliminary_assessment']} (conf={row['preliminary_confidence']})")
    print(f"Updated Assessment:    {row['updated_assessment']} (conf={row['updated_confidence']})")
    print(f"Correct Answer:        {row['correct_answer_idx']}. {row['correct_answer_text']}")
    print(f"Prelim Correct:        {row['is_correct_preliminary']}")
    print(f"Updated Correct:       {row['is_correct_updated']}")
    print(f"Confidence Delta:      {row['confidence_delta']:+d}")